In [1]:
import pandas as pd
import soccerdata as sd

print("Starting scrape...", flush=True)

leagues = [
    "ENG-Premier League",
    "ESP-La Liga", 
    "ITA-Serie A",
    "GER-Bundesliga",
    "FRA-Ligue 1",
]

for league in leagues:
    print(f"Trying {league}...", flush=True)
    try:
        u = sd.Understat(leagues=[league], seasons=["2025"], no_cache=True)
        df = u.read_schedule().reset_index()
        print(f"  Got {len(df)} rows", flush=True)
    except Exception as e:
        print(f"  Error: {e}", flush=True)

print("Done.", flush=True)

[09/08/26 08:31:54] INFO     No custom team name replacements found. You can configure these in       ]8;id=13450551;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=13450552;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\rhkha\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=13450558;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=13450559;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\rhkha\soccerdata\config\league_dict.json.                                    

Starting scrape...
Trying ENG-Premier League...


                    INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=13450566;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13450567;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-09-08 08:31:54] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


...

  Got 380 rows
Trying GER-Bundesliga...


[09/08/26 08:32:10] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=13450592;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13450593;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  Got 306 rows
Trying FRA-Ligue 1...


[09/08/26 08:32:14] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=13450598;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13450599;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  Got 306 rows
Done.


In [2]:
import sqlite3
import pandas as pd
import soccerdata as sd
from pathlib import Path

DB_PATH    = Path(r"C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\FIFA-WORLDCUP-PREDICTION\v4_historical_data.sqlite")
NEW_SEASON = "2526"
SCRAPE_SEASON = "2025"

TARGET_LEAGUES = [
    "ENG-Premier League",
    "ESP-La Liga",
    "ITA-Serie A",
    "GER-Bundesliga",
    "FRA-Ligue 1",
]

COLUMN_MAP = {
    "home_team"  : ["home_team", "home"],
    "away_team"  : ["away_team", "away"],
    "home_goals" : ["home_goals", "home_goal", "score_home"],
    "away_goals" : ["away_goals", "away_goal", "score_away"],
    "home_xg"    : ["home_xg", "xg_home", "xgh"],
    "away_xg"    : ["away_xg", "xg_away", "xga"],
    "is_finished": ["is_result", "finished", "status"],
}

def resolve_column(df, candidates):
    for name in candidates:
        if name in df.columns:
            return name
    return None

# Check existing data
conn = sqlite3.connect(DB_PATH)
before = pd.read_sql("SELECT season, COUNT(*) as n FROM matches_xg GROUP BY season ORDER BY season", conn)
conn.close()
print("Existing database:")
print(before.to_string(index=False))
print()

# Scrape
all_dfs = []
for league in TARGET_LEAGUES:
    print(f"Scraping {league}...", flush=True)
    u = sd.Understat(leagues=[league], seasons=[SCRAPE_SEASON], no_cache=True)
    df_raw = u.read_schedule().reset_index()
    print(f"  {len(df_raw)} rows", flush=True)
    all_dfs.append(df_raw)

df = pd.concat(all_dfs, ignore_index=True)

# Resolve columns
rename = {}
for standard, aliases in COLUMN_MAP.items():
    found = resolve_column(df, aliases)
    if found:
        rename[found] = standard

df = df.rename(columns=rename)

# Filter to finished matches
if "is_finished" in df.columns:
    df = df[df["is_finished"] == True].copy()
else:
    df = df.dropna(subset=["home_xg", "away_xg"]).copy()

# Standardise
for std, aliases in [("league",["league"]),("season",["season"]),("date",["date","datetime"])]:
    if std not in df.columns:
        found = resolve_column(df, aliases)
        if found:
            df = df.rename(columns={found: std})

required = ["league","season","date","home_team","away_team","home_goals","away_goals","home_xg","away_xg"]
df_clean = df[required].copy()
df_clean["date"]       = pd.to_datetime(df_clean["date"], errors="coerce")
df_clean["home_goals"] = pd.to_numeric(df_clean["home_goals"], errors="coerce")
df_clean["away_goals"] = pd.to_numeric(df_clean["away_goals"], errors="coerce")
df_clean["home_xg"]    = pd.to_numeric(df_clean["home_xg"],    errors="coerce")
df_clean["away_xg"]    = pd.to_numeric(df_clean["away_xg"],    errors="coerce")
df_clean = df_clean.dropna()

# Override season label to our short code
df_clean["season"] = NEW_SEASON

print(f"\nFinished matches with real xG: {len(df_clean):,}")
print(df_clean.groupby(["league","season"]).size().reset_index(name="matches").to_string(index=False))

# Append to database
conn = sqlite3.connect(DB_PATH)
df_clean.to_sql("matches_xg", conn, if_exists="append", index=False)
after = pd.read_sql("SELECT season, COUNT(*) as n FROM matches_xg GROUP BY season ORDER BY season", conn)
conn.close()

print("\nDatabase after append:")
print(after.to_string(index=False))
print(f"\n✅ {NEW_SEASON} appended successfully.")

Existing database:
season    n
  2122 1826
  2223 1826
  2324 1752
  2425 1752

Scraping ENG-Premier League...


[09/08/26 08:32:34] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=13450604;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13450605;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  380 rows
Scraping ESP-La Liga...


[09/08/26 08:32:38] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=13450610;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13450611;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  380 rows
Scraping ITA-Serie A...


...

  380 rows
Scraping GER-Bundesliga...


[09/08/26 08:32:47] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=13450622;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13450623;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  306 rows
Scraping FRA-Ligue 1...


[09/08/26 08:32:50] INFO     Saving cached data to C:\Users\rhkha\soccerdata\data\Understat          ]8;id=13450628;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=13450629;file://C:\Users\rhkha\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

  306 rows

Finished matches with real xG: 1,752
            league season  matches
ENG-Premier League   2526      380
       ESP-La Liga   2526      380
       FRA-Ligue 1   2526      306
    GER-Bundesliga   2526      306
       ITA-Serie A   2526      380

Database after append:
season    n
  2122 1826
  2223 1826
  2324 1752
  2425 1752
  2526 1752

✅ 2526 appended successfully.


In [ ]:
import ssl

import urllib.request
url = "https://www.football-data.co.uk/mmz4281/1415/E0.csv"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE
with urllib.request.urlopen(req, context=ssl_context, timeout=10) as resp:
    raw = resp.read().decode("utf-8", errors="replace")
lines = raw.strip().split("\n")
print(f"Lines: {len(lines)}")
print(f"Headers: {lines[0][:300]}")
print(f"Row 1: {lines[1][:300]}")

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>